In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent.parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

from ocr_vs_vlm.results_final.shared.colors import (
    get_model_color, APPROACH_COLORS, MODEL_COLORS
)
from ocr_vs_vlm.results_final.shared.stats_utils import (
    bootstrap_ci, wilcoxon_test, cohens_d, effect_size_interpretation
)
from ocr_vs_vlm.results_final.shared.viz_utils import setup_plotly_template
from ocr_vs_vlm.results_final.shared.data_loader import (
    load_dataset_data, extract_models_from_columns
)

setup_plotly_template()

DATASET = 'ICDAR_mini'
print(f"✓ Setup complete for {DATASET}")

✓ Setup complete for ICDAR_mini


## 1. Load Data

In [2]:
try:
    df = load_dataset_data(DATASET, task_type='parsing')
    print(f"✓ Loaded {len(df)} samples")
    print(f"  Approaches: {df['approach'].unique().tolist()}")
except Exception as e:
    print(f"Error loading data: {e}")
    df = pd.DataFrame()

models = extract_models_from_columns(df) if len(df) > 0 else []
print(f"  Models: {models}")

Error loading data: load_dataset_data() got an unexpected keyword argument 'task_type'
  Models: []


## 2. CER by Approach and Model

In [3]:
# Compute CER stats
stats = []
for approach in df['approach'].unique() if len(df) > 0 else []:
    app_df = df[df['approach'] == approach]
    for model in models:
        col = f'cer_{model}'
        if col in app_df.columns:
            scores = app_df[col].dropna().values
            if len(scores) > 0:
                mean, ci_lo, ci_hi = bootstrap_ci(scores, 'mean')
                stats.append({
                    'Approach': approach,
                    'Model': model,
                    'CER': mean,
                    'CI_Lower': ci_lo,
                    'CI_Upper': ci_hi,
                    'Std': np.std(scores),
                    'N': len(scores)
                })

stats_df = pd.DataFrame(stats)
if len(stats_df) > 0:
    display(Markdown("### CER Summary (Lower is Better)"))
    pivot = stats_df.pivot_table(index='Model', columns='Approach', values='CER').round(4)
    display(pivot)

In [4]:
# Visualization: CER by model
if len(stats_df) > 0:
    fig = go.Figure()
    
    stats_df_sorted = stats_df.sort_values('CER')
    
    fig.add_trace(go.Bar(
        x=[f"{row['Model']} ({row['Approach']})" for _, row in stats_df_sorted.iterrows()],
        y=stats_df_sorted['CER'],
        error_y=dict(
            type='data',
            array=stats_df_sorted['CI_Upper'] - stats_df_sorted['CER'],
            arrayminus=stats_df_sorted['CER'] - stats_df_sorted['CI_Lower']
        ),
        marker_color=[get_model_color(m) for m in stats_df_sorted['Model']],
        text=stats_df_sorted['CER'].round(3),
        textposition='outside'
    ))
    
    fig.update_layout(
        title=f'{DATASET}: Character Error Rate (CER ↓)',
        yaxis_title='CER',
        xaxis_title='Model (Approach)',
        height=500,
        xaxis_tickangle=-45
    )
    fig.show()

## 3. OCR vs VLM Comparison

In [5]:
# Aggregate by approach type
if len(df) > 0:
    approach_scores = {}
    
    for approach in df['approach'].unique():
        app_df = df[df['approach'] == approach]
        all_scores = []
        for model in models:
            col = f'cer_{model}'
            if col in app_df.columns:
                all_scores.extend(app_df[col].dropna().tolist())
        approach_scores[approach] = np.array(all_scores)
    
    print("📊 Approach Comparison")
    print("=" * 50)
    for approach, scores in approach_scores.items():
        if len(scores) > 0:
            mean, ci_lo, ci_hi = bootstrap_ci(scores, 'mean')
            print(f"\n{approach}:")
            print(f"   Mean CER: {mean:.4f} [{ci_lo:.4f}, {ci_hi:.4f}]")
            print(f"   N: {len(scores)}")

In [6]:
# Statistical test
if len(approach_scores) >= 2:
    approaches = list(approach_scores.keys())
    if len(approaches) >= 2:
        a1, a2 = approaches[0], approaches[1]
        if len(approach_scores[a1]) > 0 and len(approach_scores[a2]) > 0:
            n = min(len(approach_scores[a1]), len(approach_scores[a2]))
            stat, p = wilcoxon_test(approach_scores[a1][:n], approach_scores[a2][:n])
            d = cohens_d(approach_scores[a1], approach_scores[a2])
            
            m1, m2 = np.mean(approach_scores[a1]), np.mean(approach_scores[a2])
            winner = a1 if m1 < m2 else a2  # Lower CER is better
            
            print(f"\n📈 Statistical Test: {a1} vs {a2}")
            print(f"   Mean CER: {m1:.4f} vs {m2:.4f}")
            print(f"   Winner: {winner} (lower is better)")
            print(f"   Wilcoxon p-value: {p:.4f} {'(significant)' if p < 0.05 else '(not significant)'}")
            print(f"   Cohen's d: {d:.3f} ({effect_size_interpretation(d)})")

NameError: name 'approach_scores' is not defined

## 4. Error Distribution

In [ ]:
# Distribution of CER values
if len(df) > 0:
    fig = go.Figure()
    
    for model in models:
        col = f'cer_{model}'
        if col in df.columns:
            scores = df[col].dropna()
            if len(scores) > 0:
                fig.add_trace(go.Histogram(
                    x=scores,
                    name=model,
                    marker_color=get_model_color(model),
                    opacity=0.7
                ))
    
    fig.update_layout(
        title=f'{DATASET}: CER Distribution by Model',
        xaxis_title='Character Error Rate',
        yaxis_title='Count',
        barmode='overlay',
        height=450
    )
    fig.show()

## 5. Key Findings

In [ ]:
print("=" * 60)
print(f"{DATASET} KEY FINDINGS")
print("=" * 60)

if len(stats_df) > 0:
    best = stats_df.loc[stats_df['CER'].idxmin()]
    print(f"\n🏆 Best Configuration:")
    print(f"   {best['Approach']} + {best['Model']}: CER = {best['CER']:.4f}")
    
    approach_means = stats_df.groupby('Approach')['CER'].mean().sort_values()
    print(f"\n📊 Approach Ranking (Lower is Better):")
    for i, (app, score) in enumerate(approach_means.items(), 1):
        print(f"   {i}. {app}: {score:.4f}")

    print("\n💡 Observations:")
    print("   - ICDAR contains scene text which is challenging for both approaches")
    print("   - OCR typically performs well on printed text regions")
    print("   - VLMs may struggle with unusual fonts or orientations")

print("\n" + "=" * 60)